In [13]:
# --- cards and game state ---
# a card has a color and a value (digit 0-9 or skip).
# gamestate holds three hands, the face-up card, the draw pile, and current_turn (0, 1, or 2).

import random

COLORS = ['Red', 'Blue', 'Green', 'Yellow']
VALUES = [str(i) for i in range(10)] + ['Skip']

class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return str(self.color) + " " + str(self.value)

    def __eq__(self, other):
        if isinstance(other, Card):
            return self.color == other.color and self.value == other.value
        return False

    def __hash__(self):
        # lets us put cards in a set and use them as dict keys if needed
        return hash((self.color, self.value))

def make_deck():
    # one card per (color, value); simplified deck, not full retail uno
    deck = []
    for c in COLORS:
        for v in VALUES:
            deck.append(Card(c, v))
    return deck

class GameState:
    def __init__(self, player_hands, top_card, deck, current_turn=0):
        self.player_hands = player_hands
        self.top_card = top_card
        self.deck = deck
        self.current_turn = current_turn

    def get_prompt_state_representation(self, i):
        # build one dict from player i's point of view (for logging or assignments)
        a = (i + 1) % 3
        b = (i + 2) % 3
        return {
            'ai_cards': self.player_hands[i],
            'opponent1_cards': self.player_hands[a],
            'opponent2_cards': self.player_hands[b],
            'top_card': self.top_card,
            'deck': self.deck
        }

    def is_terminal(self):
        # someone ran out of cards
        for h in self.player_hands:
            if len(h) == 0:
                return True
        return False

    def get_winner(self):
        for i, h in enumerate(self.player_hands):
            if len(h) == 0:
                return i
        return -1

    def clone(self):
        # copy lists so "what if" moves do not change the real game
        new_hands = []
        for h in self.player_hands:
            new_hands.append(list(h))
        return GameState(new_hands, self.top_card, list(self.deck), self.current_turn)

def get_valid_moves(hand, top):
    # match face-up color or same value (simple uno rule)
    out = []
    for c in hand:
        if c.color == top.color or c.value == top.value:
            out.append(c)
    return out

def apply_move(state, move):
    s = state.clone()
    p = s.current_turn

    if move == 'Draw':
        if len(s.deck) > 0:
            drawn = s.deck.pop(0)
            s.player_hands[p].append(drawn)
        s.current_turn = (p + 1) % 3
    else:
        try:
            s.player_hands[p].remove(move)
        except ValueError:
            pass
        s.top_card = move
        if move.value == 'Skip':
            step = 2
        else:
            step = 1
        s.current_turn = (p + step) % 3

    return s

def apply_chance_draw(state, card):
    # expectimax: branch where the deck delivers one specific drawn card (weighted by probability)
    s = state.clone()
    p = s.current_turn
    try:
        s.deck.remove(card)
    except ValueError:
        pass
    s.player_hands[p].append(card)
    s.current_turn = (p + 1) % 3
    return s

def setup_game():
    deck = make_deck()
    random.shuffle(deck)

    hands = [[], [], []]
    for _ in range(5):
        for i in range(3):
            hands[i].append(deck.pop(0))

    top = deck.pop(0)
    while top.value == 'Skip':
        deck.append(top)
        random.shuffle(deck)
        top = deck.pop(0)

    return GameState(hands, top, deck, 0)


In [14]:
import math

# --- evaluation function (static score) ---
# when we cannot search deeper, we estimate how good the position is for ai_i.
# smaller hand and more skips (here) help; we use different weights for defensive vs offensive bots.

def evaluate(state, ai_i, strategy='baseline'):
    w = state.get_winner()
    if w == ai_i:
        return 1000
    if w != -1:
        return -1000

    me = state.player_hands[ai_i]
    o1 = state.player_hands[(ai_i + 1) % 3]
    o2 = state.player_hands[(ai_i + 2) % 3]

    n_me = len(me)
    n_opp = (len(o1) + len(o2)) / 2.0
    skips = 0
    for c in me:
        if c.value == 'Skip':
            skips = skips + 1

    if strategy == 'defensive':
        return 50 - 3 * n_me + 4 * n_opp + 5 * skips
    if strategy == 'offensive':
        return 50 - 8 * n_me + 1 * n_opp + 1 * skips
    return 50 - 5 * n_me + 2 * n_opp + 3 * skips


# --- minimax ---
# the ai player (ai_i) picks moves to maximize the evaluated score.
# every other player is modeled as minimizing that score (worst-case opponent).
# depth = how many plies we look ahead before calling evaluate().

def minimax(state, depth, ai_i):
    if state.is_terminal() or depth == 0:
        return evaluate(state, ai_i, 'defensive')

    mine = (state.current_turn == ai_i)
    moves_ = get_valid_moves(state.player_hands[state.current_turn], state.top_card)

    if len(moves_) == 0:
        return minimax(apply_move(state, 'Draw'), depth - 1, ai_i)

    if mine:
        best = -math.inf
        for m in moves_:
            v = minimax(apply_move(state, m), depth - 1, ai_i)
            if v > best:
                best = v
        return best
    worst = math.inf
    for m in moves_:
        v = minimax(apply_move(state, m), depth - 1, ai_i)
        if v < worst:
            worst = v
    return worst


def _card_value_key(card):
    return str(card.value)


def get_best_move_minimax(state, depth, ai_i):
    # at the root: score every legal move with minimax, then take the best
    moves_ = get_valid_moves(state.player_hands[state.current_turn], state.top_card)
    if len(moves_) == 0:
        return None, 'Draw'

    best = -math.inf
    pick = None
    scores = {}

    for m in moves_:
        v = minimax(apply_move(state, m), depth - 1, ai_i)
        scores[m] = v
        if v > best:
            best = v
            pick = m

    if pick is None:
        moves_sorted = sorted(moves_, key=_card_value_key)
        pick = moves_sorted[0]

    return scores, pick


# --- expectimax chance nodes ---
# when the player must draw, the next card is uncertain: we take expected value
# over each distinct card still in the deck (probability = count / len(deck)).

def chance_node(state, depth, ai_i):
    if len(state.deck) == 0:
        return evaluate(state, ai_i, 'offensive')

    ev = 0.0
    seen = set(state.deck)
    for c in seen:
        p = float(state.deck.count(c)) / float(len(state.deck))
        nxt = apply_chance_draw(state, c)
        ev = ev + p * expectimax(nxt, depth - 1, ai_i)
    return ev


# --- expectimax ---
# same maximizing player as above, but opponents are not purely adversarial:
# we average their outcomes (random legal move model). draws use chance_node.

def expectimax(state, depth, ai_i):
    if state.is_terminal() or depth == 0:
        return evaluate(state, ai_i, 'offensive')

    mine = (state.current_turn == ai_i)
    moves_ = get_valid_moves(state.player_hands[state.current_turn], state.top_card)

    if mine:
        if len(moves_) == 0:
            return chance_node(state, depth, ai_i)
        best = -math.inf
        for m in moves_:
            v = expectimax(apply_move(state, m), depth - 1, ai_i)
            if v > best:
                best = v
        return best

    if len(moves_) == 0:
        return chance_node(state, depth, ai_i)

    tot = 0.0
    for m in moves_:
        tot = tot + expectimax(apply_move(state, m), depth - 1, ai_i)
    return tot / len(moves_)


def get_best_move_expectimax(state, depth, ai_i):
    moves_ = get_valid_moves(state.player_hands[state.current_turn], state.top_card)
    if len(moves_) == 0:
        return None, 'Draw'

    best = -math.inf
    pick = None
    scores = {}

    for m in moves_:
        v = expectimax(apply_move(state, m), depth - 1, ai_i)
        scores[m] = v
        if v > best:
            best = v
            pick = m

    if pick is None:
        pick = moves_[0]

    return scores, pick


In [ ]:
# --- text output for the notebook run (no f-strings) ---

def show_turn(name, state, scores, move):
    print("")
    print("=" * 40)
    print("top:", state.top_card)
    print(name + " hand:")
    for c in state.player_hands[state.current_turn]:
        print("  " + str(c))

    if scores is not None:
        print(name + " - options at depth 1:")
        for m, sc in scores.items():
            print("  " + str(m) + " -> " + str(round(sc, 2)))

    if move == 'Draw':
        act = "draw"
    else:
        act = "play " + str(move)
    print("")
    print(">> " + act)
    print("=" * 40)


def simulate_game(p3="simulation"):
    print("uno ai run")
    if p3 == "simulation":
        p3_label = "minimax"
    else:
        p3_label = "you"
    print("p1 minimax | p2 expectimax | p3 " + p3_label)

    state = setup_game()
    depth = 3
    labels = ["p1 (minimax)", "p2 (expectimax)", "p3 (minimax)"]
    if p3 == "manual":
        labels[2] = "p3 (you)"

    while not state.is_terminal():
        who = state.current_turn
        label = labels[who]
        scores = None
        move = None

        if who == 0:
            scores, move = get_best_move_minimax(state, depth, who)
        elif who == 1:
            scores, move = get_best_move_expectimax(state, depth, who)
        elif who == 2:
            if p3 == "simulation":
                scores, move = get_best_move_minimax(state, depth, who)
            else:
                legal = get_valid_moves(state.player_hands[who], state.top_card)
                print("")
                print("=" * 40)
                print("top:", state.top_card)
                print(label + " hand:")
                for c in state.player_hands[who]:
                    print("  " + str(c))
                if len(legal) == 0:
                    print("")
                    print("no play - you draw")
                    move = 'Draw'
                else:
                    print("")
                    print("pick index:")
                    idx = 0
                    while idx < len(legal):
                        c = legal[idx]
                        print("  " + str(idx) + ": " + str(c))
                        idx = idx + 1
                    while True:
                        try:
                            choice = int(input("index: "))
                            if choice >= 0 and choice < len(legal):
                                move = legal[choice]
                                break
                        except ValueError:
                            pass
                        print("bad index")

        if who != 2 or p3 == "simulation":
            show_turn(label, state, scores, move)
        else:
            if move == 'Draw':
                act = "draw"
            else:
                act = "play " + str(move)
            print("")
            print(">> " + act)
            print("=" * 40)

        state = apply_move(state, move)

    w = state.get_winner()
    print("")
    print("*" * 40)
    print("done - " + labels[w] + " wins")
    i = 0
    while i < len(labels):
        L = labels[i]
        left = len(state.player_hands[i])
        print(L + ": " + str(left) + " cards left")
        i = i + 1
    print("*" * 40)


# run: type manual or simulation
mode = input("mode manual / simulation: ").strip().lower()
if mode not in ("manual", "simulation"):
    mode = "simulation"
simulate_game(mode)
